In [ ]:
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from collections import Counter

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Note: LinearRegression/Lasso/Elastic Net/Ridge did not make sense for the fraud use case (binary classification). So I instead use Logistic Reg with different penalty

In [2]:
random_state=42

In [3]:
classification_reports = []
classification_report_keys = []

## IBM

### Data

In [4]:
# Import IBM dataset
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [5]:
df.drop(columns=['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'], inplace=True)

### Undersampling

In [6]:
from collections import Counter


X = df.drop(columns='Is Laundering')
y = df['Is Laundering']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 5073168, 1: 5177})
Resampled dataset shape: Counter({0: 5177, 1: 5177})


### Split Data

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

#### Linear Regression

In [8]:
model = LogisticRegression()

In [9]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [10]:
y_pred = model.predict(X_test)

In [11]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logistic Regression')

              precision    recall  f1-score   support

           0       1.00      0.01      0.02      1036
           1       0.50      1.00      0.67      1035

    accuracy                           0.50      2071
   macro avg       0.75      0.50      0.34      2071
weighted avg       0.75      0.50      0.34      2071



#### Ridge

In [12]:
from sklearn.linear_model import RidgeClassifier


ridge_model = RidgeClassifier(random_state=random_state)

In [13]:
ridge_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.80134e-23): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,class_weight,None
,solver,'auto'
,positive,False
,random_state,42


In [14]:
y_pred = ridge_model.predict(X_test)

In [15]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Ridge Classifier')


              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



#### Lasso

In [16]:
lasso_model = LogisticRegression(penalty='l1', solver='liblinear')

In [17]:
lasso_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [18]:
y_pred = lasso_model.predict(X_test)

In [19]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline L1 (Lasso) Logistic Regression')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



### Scaling Data

In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Parameter Tuning

#### Logistic Regression

In [21]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [22]:
# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5)

In [23]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_recall_optuna_results)

[I 2026-06-24 19:11:43,261] A new study created in memory with name: no-name-aa1fbdae-f775-4056-903f-2b608112b814
[I 2026-06-24 19:11:44,856] Trial 0 finished with value: 0.8597323473365851 and parameters: {'C': 0.7971471149212764, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:11:46,055] Trial 1 finished with value: 0.8597323473365851 and parameters: {'C': 0.062296252717312005, 'Class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:11:47,447] Trial 2 finished with value: 0.8597323473365851 and parameters: {'C': 0.32511085134283163, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:11:48,851] Trial 3 finished with value: 0.8597323473365851 and parameters: {'C': 0.6465034220326928, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:11:49,025] Trial 4 finished with value: 0.8597323473365851 and parameters: {'C': 0.660404867

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
92,92,0.861423,2026-06-24 19:12:03.037365,2026-06-24 19:12:03.195615,0 days 00:00:00.158250,0.003415,balanced,COMPLETE
63,63,0.861423,2026-06-24 19:11:58.356413,2026-06-24 19:11:58.495597,0 days 00:00:00.139184,0.002048,None,COMPLETE
0,0,0.859732,2026-06-24 19:11:43.262091,2026-06-24 19:11:44.856665,0 days 00:00:01.594574,0.797147,balanced,COMPLETE
1,1,0.859732,2026-06-24 19:11:44.858329,2026-06-24 19:11:46.055537,0 days 00:00:01.197208,0.062296,None,COMPLETE
4,4,0.859732,2026-06-24 19:11:48.852619,2026-06-24 19:11:49.025572,0 days 00:00:00.172953,0.660405,balanced,COMPLETE


In [24]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_f1_optuna_results)

[I 2026-06-24 19:12:04,339] A new study created in memory with name: no-name-bf364028-9e6b-4b58-9ddf-a161dbbb5e87
[I 2026-06-24 19:12:04,506] Trial 0 finished with value: 0.8771824847322993 and parameters: {'C': 0.6670177538704194, 'Class_weight': None}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:04,658] Trial 1 finished with value: 0.8771824847322993 and parameters: {'C': 0.014561258990156301, 'Class_weight': None}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:04,805] Trial 2 finished with value: 0.8771824847322993 and parameters: {'C': 0.4705405784046112, 'Class_weight': None}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:04,977] Trial 3 finished with value: 0.8771824847322993 and parameters: {'C': 0.3804685332403221, 'Class_weight': None}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:05,147] Trial 4 finished with value: 0.8771824847322993 and parameters: {'C': 0.5652260266098672, 'Class_wei

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
11,11,0.878148,2026-06-24 19:12:06.091475,2026-06-24 19:12:06.225341,0 days 00:00:00.133866,0.001838,None,COMPLETE
96,96,0.878148,2026-06-24 19:12:19.746743,2026-06-24 19:12:19.899790,0 days 00:00:00.153047,0.002473,None,COMPLETE
2,2,0.877182,2026-06-24 19:12:04.659755,2026-06-24 19:12:04.805449,0 days 00:00:00.145694,0.470541,None,COMPLETE
0,0,0.877182,2026-06-24 19:12:04.341658,2026-06-24 19:12:04.506496,0 days 00:00:00.164838,0.667018,None,COMPLETE
3,3,0.877182,2026-06-24 19:12:04.806452,2026-06-24 19:12:04.977283,0 days 00:00:00.170831,0.380469,None,COMPLETE


##### Tryout best params

In [25]:
log_reg_recall_optimized_model = LogisticRegression(C=0.769946, class_weight='balanced', random_state=random_state)
log_reg_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



In [26]:
log_reg_f1_optimized_model = LogisticRegression(C=0.315005, class_weight='balanced', random_state=random_state)
log_reg_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



#### Ridge

In [27]:
# We can tune alpha and class weight

##### Recall optimized

In [28]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_recall_optuna_results)

[I 2026-06-24 19:12:20,495] A new study created in memory with name: no-name-668ab559-2e13-4b35-8736-844a4cb23b6a
[I 2026-06-24 19:12:20,650] Trial 0 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.6730975660407932, 'class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:20,805] Trial 1 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.097804436962455, 'class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:20,959] Trial 2 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.4500130294400889, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:21,114] Trial 3 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.9031458743315879, 'class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:21,271] Trial 4 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.43972

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
0,0,0.859732,2026-06-24 19:12:20.496397,2026-06-24 19:12:20.650930,0 days 00:00:00.154533,0.673098,None,COMPLETE
1,1,0.859732,2026-06-24 19:12:20.650930,2026-06-24 19:12:20.805622,0 days 00:00:00.154692,0.097804,None,COMPLETE
2,2,0.859732,2026-06-24 19:12:20.806757,2026-06-24 19:12:20.959898,0 days 00:00:00.153141,0.450013,balanced,COMPLETE
3,3,0.859732,2026-06-24 19:12:20.960900,2026-06-24 19:12:21.114783,0 days 00:00:00.153883,0.903146,None,COMPLETE
4,4,0.859732,2026-06-24 19:12:21.115783,2026-06-24 19:12:21.271082,0 days 00:00:00.155299,0.439723,balanced,COMPLETE


##### Tryout best params

In [29]:
ridge_recall_optimized_model = RidgeClassifier(alpha=0.039558, class_weight='balanced', random_state=random_state)
ridge_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



##### F1 optimized

In [30]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_f1_optuna_results)

[I 2026-06-24 19:12:36,039] A new study created in memory with name: no-name-6c11b74c-ceca-40c8-a20a-bd91ae168e5c
[I 2026-06-24 19:12:36,200] Trial 0 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.689934173857699, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-06-24 19:12:36,361] Trial 1 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.7965228709472622, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-06-24 19:12:36,508] Trial 2 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.4382108651422037, 'class_weight': None}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-06-24 19:12:36,649] Trial 3 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.3420221630051275, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-06-24 19:12:36,804] Trial 4 finished with value: 0.8769665578707571 and parameters: {'alp

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
0,0,0.876967,2026-06-24 19:12:36.040531,2026-06-24 19:12:36.200983,0 days 00:00:00.160452,0.689934,balanced,COMPLETE
1,1,0.876967,2026-06-24 19:12:36.201985,2026-06-24 19:12:36.361259,0 days 00:00:00.159274,0.796523,balanced,COMPLETE
2,2,0.876967,2026-06-24 19:12:36.362260,2026-06-24 19:12:36.508678,0 days 00:00:00.146418,0.438211,None,COMPLETE
3,3,0.876967,2026-06-24 19:12:36.509670,2026-06-24 19:12:36.649810,0 days 00:00:00.140140,0.342022,balanced,COMPLETE
4,4,0.876967,2026-06-24 19:12:36.650810,2026-06-24 19:12:36.804060,0 days 00:00:00.153250,0.360593,None,COMPLETE


##### Tryout best params

In [31]:
ridge_f1_optimized_model = RidgeClassifier(alpha=0.364826, class_weight='balanced', random_state=random_state)
ridge_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



#### Lasso Logistic Regression

In [32]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [33]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15, n_jobs=-1)

lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_recall_optuna_results)

[I 2026-06-24 19:12:51,343] A new study created in memory with name: no-name-5a8dc142-81ff-41cf-add3-8d5b9a676055
[I 2026-06-24 19:12:51,663] Trial 0 finished with value: 0.8597323473365851 and parameters: {'C': 0.07036797644008005, 'Class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:51,890] Trial 3 finished with value: 0.8597323473365851 and parameters: {'C': 0.14525305415291648, 'Class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:52,060] Trial 4 finished with value: 0.8597323473365851 and parameters: {'C': 0.09409652652837042, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:52,304] Trial 2 finished with value: 0.8597323473365851 and parameters: {'C': 0.4861290807464944, 'Class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-06-24 19:12:52,308] Trial 1 finished with value: 0.8597323473365851 and parameters: {'C': 0.6843107347808559, 'Cl

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
0,0,0.859732,2026-06-24 19:12:51.344571,2026-06-24 19:12:51.663344,0 days 00:00:00.318773,0.070368,None,COMPLETE
1,1,0.859732,2026-06-24 19:12:51.346780,2026-06-24 19:12:52.308409,0 days 00:00:00.961629,0.684311,None,COMPLETE
2,2,0.859732,2026-06-24 19:12:51.348286,2026-06-24 19:12:52.304581,0 days 00:00:00.956295,0.486129,None,COMPLETE
3,3,0.859732,2026-06-24 19:12:51.350391,2026-06-24 19:12:51.890184,0 days 00:00:00.539793,0.145253,None,COMPLETE
4,4,0.859732,2026-06-24 19:12:51.353392,2026-06-24 19:12:52.060142,0 days 00:00:00.706750,0.094097,balanced,COMPLETE


In [34]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_f1_optuna_results)

[I 2026-06-24 19:12:52,609] A new study created in memory with name: no-name-e69eff02-d889-4033-9ebc-4b89d0aa2e58
[I 2026-06-24 19:12:52,853] Trial 0 finished with value: 0.8771824847322993 and parameters: {'C': 0.4450858226672189, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:53,103] Trial 1 finished with value: 0.8771824847322993 and parameters: {'C': 0.47373705419000345, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:53,381] Trial 2 finished with value: 0.8771824847322993 and parameters: {'C': 0.834919925338008, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:53,586] Trial 3 finished with value: 0.8771824847322993 and parameters: {'C': 0.14639735881640423, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-06-24 19:12:53,789] Trial 4 finished with value: 0.8771824847322993 and parameters: {'C': 0.24827

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
0,0,0.877182,2026-06-24 19:12:52.609233,2026-06-24 19:12:52.853672,0 days 00:00:00.244439,0.445086,balanced,COMPLETE
1,1,0.877182,2026-06-24 19:12:52.854672,2026-06-24 19:12:53.103783,0 days 00:00:00.249111,0.473737,balanced,COMPLETE
2,2,0.877182,2026-06-24 19:12:53.103783,2026-06-24 19:12:53.381007,0 days 00:00:00.277224,0.834920,balanced,COMPLETE
3,3,0.877182,2026-06-24 19:12:53.382008,2026-06-24 19:12:53.586640,0 days 00:00:00.204632,0.146397,balanced,COMPLETE
4,4,0.877182,2026-06-24 19:12:53.587680,2026-06-24 19:12:53.789747,0 days 00:00:00.202067,0.248272,None,COMPLETE


##### Tryout best params

In [35]:
lasso_recall_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.248927, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [36]:
lasso_f1_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.496023, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Conclusions

In [37]:
pd.concat(classification_reports, keys=classification_report_keys)

precision    recall  \
Baseline Logistic Regression            0              1.000000  0.007722   
                                        1              0.501697  1.000000   
                                        accuracy       0.503621  0.503621   
                                        macro avg      0.750848  0.503861   
                                        weighted avg   0.750969  0.503621   
Baseline Ridge Classifier               0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Baseline L1 (Lasso) Logistic Regression 0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Logistic Regression (Recall Optimized)  0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Logistic Regression (F1 Optimized)      0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Ridge Classifier (Recall Optimized)     0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Ridge Classifier (F1 Optimized)         0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Lasso Classifier (Recall Optimized)     0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Lasso Classifier (F1 Optimized)         0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   

                                                      f1-score      support  
Baseline Logistic Regression            0             0.015326  1036.000000  
                                        1             0.668173  1035.000000  
                                        accuracy      0.503621     0.503621  
                                        macro avg     0.341749  2071.000000  
                                        weighted avg  0.341592  2071.000000  
Baseline Ridge Classifier               0  

The lasso and ridge both overcome poor initial performance of the logistic regression model

The logistic reg performance could have been poor due to severe multicollinearity or imbalanced feature scaling

In [38]:
import numpy as np

pd.set_option('display.max_rows', None)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Ridge_Coef': ridge_f1_optimized_model.coef_,
})

# Sort features by the absolute size of their coefficients
coef_df['Abs_Coefficient'] = np.abs(coef_df['Ridge_Coef'])
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)
coef_df

,Feature,Ridge_Coef,Abs_Coefficient
94,Payment Format_ACH,0.439854,0.439854
97,Payment Format_Cheque,-0.244206,0.244206
101,Account_Same,-0.235530,0.235530
98,Payment Format_Credit Card,-0.225024,0.225024
72,Receiving Currency_Saudi Riyal,0.203327,0.203327
87,Payment Currency_Saudi Riyal,-0.165397,0.165397
100,Payment Format_Wire,-0.123514,0.123514
96,Payment Format_Cash,-0.118390,0.118390
74,Receiving Currency_Swiss Franc,-0.108524,0.108524
89,Payment Currency_Swiss Franc,0.095514,0.095514


In [39]:
len(coef_df[coef_df['Abs_Coefficient'] == 0])

60

We can see 60 features coefs get shrunk to 0 by the Ridge model. These all consist of the _Mean and _Median computed features which aimed to give a summary of average transactions for a given currency. This shows these features did not add additional information to the model and can be safely dropped

In [40]:
coefficients = lasso_f1_optimized_model.coef_[0]

# 3. Create a DataFrame to map features to their coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients
})

# 4. VIEW RETAINED FEATURES (Coefficient is NOT 0)
selected_features = coef_df[coef_df['Coefficient'] != 0]
print("Features Selected by Lasso:")
print(selected_features)

# 5. VIEW REJECTED FEATURES (Coefficient is exactly 0)
dropped_features = coef_df[coef_df['Coefficient'] == 0]
print("\nFeatures Ignored by Lasso:")
print(dropped_features['Feature'].tolist())

Features Selected by Lasso:
                                  Feature  Coefficient
0                         Amount Received     0.017789
1                             Amount Paid     0.017961
2                     Amount_Received_USD    -0.016400
3                         Amount_Paid_USD    -0.016484
64   Receiving Currency_Australian Dollar    -0.038141
65             Receiving Currency_Bitcoin    -0.011198
66         Receiving Currency_Brazil Real    -0.017142
67     Receiving Currency_Canadian Dollar    -0.053279
68                Receiving Currency_Euro     0.033876
69        Receiving Currency_Mexican Peso    -0.008931
70               Receiving Currency_Ruble    -0.007914
71               Receiving Currency_Rupee    -0.014150
72         Receiving Currency_Saudi Riyal     0.141448
73              Receiving Currency_Shekel    -0.074872
74         Receiving Currency_Swiss Franc    -0.049205
75            Receiving Currency_UK Pound    -0.023084
76           Receiving Currency_US Do

We can see Lasso also removed some features during feature selection. The features consist of the same summary statistic features which Ridge shrunk the coefficients of.